---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: Web Analytics

### 📋 **Topic**: Using LLMs

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.



---

In this Lecture, we'll go over many tools that will help us use LLMs more easily and effectively in our projects. In particular, we will cover:

1. Using Async Programming to speed things up
2. making our LLM calls robust to errors and rate limits with retries
3. Logging, tracing, and experiment tracking. Even a lightweight version is huge: Log: prompt, model, params, response, latency, cost estimate. Use something like structured logging or a tiny wrapper that returns a record object. Optionally mention tools (Weights & Biases, plain SQLite, CSV) to track experiments. The message: “If you don’t log it, you can’t debug or improve it.” 
4. 
5. using Pydantic for structured outputs

Optional:
1. using `llama-index` for simple RAG tasks

## 1. Using Async Programming to speed things up

Instead of sending prompts one by one, you can send multiple prompts asynchronously.
- We'll see what this means by example.

Let's say you have 10 prompts you want to process.

In [ ]:
import openai
import os
import time # we'll use this to time our operations
from dotenv import load_dotenv 

load_dotenv() # expose our API key from our .env as an environment variable
openai.api_key = os.getenv("OPENAI_API_KEY")

prompts = [
    "What is the capital of France?",
    "What is the capital of Greece?",
    "What is the capital of Bulgaria?",
    "What is the capital of Panama?",
    "What is the capital of Pakistan?",
    "What is the capital of Egypt?",
    "What is the capital of India?",
    "What is the capital of China?",
    "What is the capital of Brazil?",
    "What is the capital of Argentina?",
    "What is the capital of Mexico?"
]



What we did last week, was to wait for each response to come back before sending the next one.

This is called **sequential processing**.

In [ ]:
# Inefficient: sending one by one
start_time = time.time()
results_one_by_one = []
for prompt in prompts:
    response = openai.chat.completions.create(
        model="gpt-4.1",
        messages=[{"role": "user", "content": prompt}]
    )
    results_one_by_one.append(response.choices[0].message.content)
    print(f"Prompt: {prompt}")
    print(f"Response: {response.choices[0].message.content}\n")
end_time = time.time()
print(f"Sequential processing took: {end_time - start_time:.2f} seconds")


When we write code normally (called **synchronous/sequential code**), Python does one thing at a time:
- Send a request
- Wait for response… 
- Response arrives
- Move to the next request

This is simple, but slow — especially when each request takes 1–2 seconds. Examples of slow requests are:
- Reading from a database
- Reading from a file
- **Making an API call**
  
**Async** (short for asynchronous) means that we can start many independent tasks at the same time.
- We will wait for all of them to finish, but we can start them all at once.
- This is perfect for doing things such as calling the OpenAI API, because the model is doing most of the work, while your computer is just waiting


When a function is marked with:

```python
async def ...
```

it means that
- The function can be paused while it waits (for example, waiting for the OpenAI API to respond)
- Python can move on to other tasks while the function is paused
- Later, when the response comes back, Python continues the function

Inside async functions, we use:
```python
await ...
```
to pause the function until the awaited task is finished.





In [ ]:
import asyncio # This is a library that allows us to run code asynchronously
from openai import AsyncOpenAI # This is the async version of the OpenAI API

client = AsyncOpenAI()

# note that this function definition begins with "async"
# this means that the function can be run asynchronously
async def get_completion(prompt: str):
    resp = await client.chat.completions.create(
        model="gpt-4.1",
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content


async def get_completions(prompts):
    start = time.time()

    # Fire all requests concurrently
    tasks = [asyncio.create_task(get_completion(prompt)) for prompt in prompts]
    results = await asyncio.gather(*tasks)

    # Print results
    for prompt, answer in zip(prompts, results):
        print(f"Prompt: {prompt}")
        print(f"Response: {answer}\n")

    end = time.time()
    print(f"Concurrent processing took: {end - start:.2f} seconds")

    return results


In [ ]:
await get_completions(prompts)



# 5. LlamaIndex

## 5.1 Overhead

Now that we have gained some familiarity with the OpenAI API, we'll learn how to use the `llama-index` package.
- `llama-index` relies on the power of LLMs to perform RAG (retrieval-augmented generation) tasks
- Plainly, this means imbuing LLMs with knowledge from our own data


In [ ]:
from llama_index.core.indices.vector_store.base import VectorStoreIndex
from llama_index.core import (
  SimpleDirectoryReader,
  StorageContext,
  load_index_from_storage,
)

## 5.2 Question answering

**How does Q&A with `llama-index` work?**

**Step 1: Loading**

First, we will load our document from file using the `SimpleDirectoryReader`.

To answer questions about our document, `llama-index` needs to convert the document into a format that allows for semantic search. This involves breaking the document up into smaller sections called nodes.

**Step 2: Indexing**

Then, we create a `VectorStoreIndex`. 

Then, `llama-index` calls the OpenAI embeddings API to convert the text of these nodes into contextual embeddings (vectors with floats), just like we saw in the previous section. By default, it uses the text-embedding-ada-002 model.

**Step 3: Persisting**

Calling the embeddings API costs money and relies on your OpenAI API key. Therefore, we call `index.storage_context.persist`, which stores our index to disk so that we don't have to reindex our document more than once.


We'll start by loading and indexing a simple job posting. We'll also save this index to disk, so we don't need to re-index it later.

In [ ]:
# put the file here
data_filepath = "PATH/TO/YOUR/DIRECTORY"
# llama index will create this folder
storage_dir = "storage"

# Check if storage already exists
if not os.path.exists(storage_dir):
  
  # Load the documents and create the index
  documents = SimpleDirectoryReader(data_filepath).load_data()
  index = VectorStoreIndex.from_documents(documents, show_progress=True)

  # Store it for later
  index.storage_context.persist()

else:
    
  storage_context = StorageContext.from_defaults(persist_dir=storage_dir)
  index = load_index_from_storage(storage_context)

# Create a query engine
query_engine = index.as_query_engine()

**Step 4: Querying**

Now comes the fun part: querying! Let's ask some questions about our document.

In [ ]:
response = query_engine.query("What programming languages do I need to know for this job?")
print(response)

In [ ]:
response = query_engine.query("What degree is required for this job?")
print(response)

In [ ]:
response = query_engine.query("How many weeks will I be working for?")
print(response)

## 5.3 Extracting structured data

In [ ]:
# pip install llama-index-program-evaporate

In [ ]:
from llama_index.program.openai import OpenAIPydanticProgram
from llama_index.program.evaporate.df import DFRowsProgram

Here, we have a large block of text with several job listings. Let's use `llama-index` to extract relevant data in a structured format.

In [ ]:
text = """
Adobe is seeking talented and passionate Software Engineer interns across all organizations.
All 2024 Adobe interns will be co-located hybrid.
You need proficiency and experience with the following: Java, C++, JavaScript, Python.
The U.S. pay range for this position is $45.00 -- $55.00 hourly.

At Amazon, we hire the best minds in technology to innovate and build on behalf of our customers. 
Programming experience with at least one modern language such as Java, C++, or C# including object-oriented design is required.
The base pay for a Software Development Engineer Intern ranges from $42.50/hr in our lowest geographic market up to $96.15/hr in our highest geographic market.
The majority of our SDE roles are based in the greater Seattle/Bellevue, WA area.


Zoox is looking for a system engineering intern to join our Systems Design and Mission Assurance (SDMA) team. 
You need experience working with vehicle dynamics simulation tools such as Carmaker and/or Carsim, as well as experience with Python.
You will work with a cross functional team in Foster City, CA to evaluate the autonomous vehicle’s response to various electrical/mechanical faults in the motion control system.
We are data-driven, transparent, and consistent. The target rate for this role is $50-$74.51/hr.
"""

Initialize an empty dataframe containing the fields you want to extract from the text, along with their data types.

In [ ]:
df = pd.DataFrame(
    {
        "Employer": pd.Series(dtype="str"),
        "Position": pd.Series(dtype="str"),
        "Python Required": pd.Series(dtype="bool"),
        "C++ Required": pd.Series(dtype="bool")
    }
)

In [ ]:
# Initialize a program that will extract rows from the text, using your existing dataframe schema
df_rows_program = DFRowsProgram.from_defaults(
    pydantic_program_cls=OpenAIPydanticProgram, df=df
)

# Use the program to parse the text and generate rows
result_obj = df_rows_program(input_str=text)

Now, we turn the results into a dataframe so we can explore the data and perform analyses.

In [ ]:
# Create a dataframe from our result object
dataframe_rows = []
for row in result_obj.rows:
  dataframe_rows.append(row.row_values)
  
jobs = pd.DataFrame(dataframe_rows, columns=["Employer", "Position", "Python Required", "C++ Required"])

jobs

In [ ]:
# Convert "Yes"/"No" to True/False if column names are correct
jobs["Python Required"] = jobs["Python Required"].map({"Yes": True, "No": False})
jobs["C++ Required"] = jobs["C++ Required"].map({"Yes": True, "No": False})

In [ ]:
jobs[jobs["Python Required"] & ~jobs["C++ Required"]]


In [ ]:
# jobs_filtered = jobs[(jobs["Python Required"] == True) & (jobs["C++ Required"] != True)]
# jobs_filtered = jobs[(jobs["Python Required"] == "Yes") & (jobs["C++ Required"] != "Yes")]

In [ ]:
# jobs[jobs["Python Required"] & ~jobs["C++ Required"]]

## 5.4 Unsupervised data extraction

In [ ]:
from llama_index.program.evaporate.df import DFFullProgram

You can also do something pretty awesome with `llama-index`. If you don't want to specify the dataframe schema manually, you can leave it up to the LLM to figure out what to extract by running `DFFullProgram` instead of `DfRowsProgram`.

In [ ]:
df_full_program = DFFullProgram.from_defaults(
    pydantic_program_cls=OpenAIPydanticProgram,
)

result_obj = df_full_program(input_str=text)

In [ ]:
jobs = result_obj.to_df()

jobs

Because the dataframe schema is generated by the LLM, it might take a little more work to get the data into a usable format, but this is still a lot more efficient than going through the data yourself.

In [ ]:
hourly_rate_cols = ["Min Hourly Rate", "Max Hourly Rate"]

jobs[hourly_rate_cols] = jobs["Pay Range"].str.extract(r"\$(\d+\.\d+) - \$(\d+\.\d+)")

jobs[hourly_rate_cols] = jobs[hourly_rate_cols].astype(float)

jobs.sort_values(by="Max Hourly Rate", ascending=False)

## 5.5 Querying pandas

In [ ]:
# pip install llama-index llama-index-experimental

In [ ]:
from llama_index.experimental.query_engine import PandasQueryEngine

In [ ]:
query_engine = PandasQueryEngine(df=jobs, verbose=True)

You can also use `llama-index` can also convert your natural language queries directly into pandas commands and run those commands on your dataframe. This might come in handy if you were trying to build an application like a data chatbot!

In [ ]:
response = query_engine.query("What is the max hourly rate of jobs that require Python?")

print(f"The max hourly rate is: ${response}")